# 图像梯度计算与经典边缘检测实验练习版

本 notebook 是学生练习版本。你需要补全关键函数，实现 Sobel、Prewitt、Roberts、拉普拉斯算子的梯度或边缘响应计算，并实现经典 Canny 边缘检测流程。

要求：

- 自己实现二维卷积函数。
- 自己实现 Sobel、Prewitt、Roberts 和拉普拉斯算子。
- 自己实现 Canny 边缘检测的关键步骤。
- 绘图标题使用英文。
- 默认优先读取当前目录下的 Lena 图像文件。

## 相关知识

图像边缘通常对应灰度变化剧烈的位置。设灰度图像为 $f(x, y)$，一阶梯度可以表示为：

$$\nabla f = \left[\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}\right] = [G_x, G_y]$$

梯度幅值和方向为：

$$M(x, y)=\sqrt{G_x^2+G_y^2}$$

$$\theta(x, y)=\arctan2(G_y, G_x)$$

### 1. Sobel 算子

Sobel 算子用两个 $3 \times 3$ 模板估计 x 和 y 方向梯度，并带有一定平滑作用。

$$G_x = \begin{bmatrix}-1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1\end{bmatrix} * f$$

$$G_y = \begin{bmatrix}-1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1\end{bmatrix} * f$$

### 2. Prewitt 算子

Prewitt 算子与 Sobel 类似，但模板权重更均匀。

$$G_x = \begin{bmatrix}-1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1\end{bmatrix} * f$$

$$G_y = \begin{bmatrix}-1 & -1 & -1 \\ 0 & 0 & 0 \\ 1 & 1 & 1\end{bmatrix} * f$$

### 3. Roberts 算子

Roberts 算子使用 $2 \times 2$ 交叉差分模板，对细小边缘敏感，但也更容易受噪声影响。

$$G_x = \begin{bmatrix}1 & 0 \\ 0 & -1\end{bmatrix} * f$$

$$G_y = \begin{bmatrix}0 & 1 \\ -1 & 0\end{bmatrix} * f$$

### 4. 拉普拉斯算子

拉普拉斯算子是二阶微分算子：

$$\nabla^2 f=\frac{\partial^2 f}{\partial x^2}+\frac{\partial^2 f}{\partial y^2}$$

4 邻域模板：

$$\begin{bmatrix}0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0\end{bmatrix}$$

8 邻域模板：

$$\begin{bmatrix}1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1\end{bmatrix}$$

### 5. Canny 边缘检测

Canny 的基本流程为：

1. 高斯平滑：使用高斯核降低噪声。
2. 梯度计算：通常使用 Sobel 算子得到 $G_x$、$G_y$、梯度幅值和方向。
3. 非极大值抑制：沿梯度方向保留局部最大值，使边缘变细。
4. 双阈值检测：分出强边缘、弱边缘和非边缘。
5. 滞后边缘连接：保留与强边缘连通的弱边缘，删除孤立弱边缘。

## 公共代码

请先补全 `convolve2d`，后续所有算子都依赖这个函数。

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, img_as_float, io
from skimage.color import rgb2gray, rgba2rgb


def load_gray_image():
    """优先读取当前目录下的 Lena 图像；如果没有，则使用内置人像图作为备用。"""
    candidate_paths = [
        Path("lena.png"), Path("lena.jpg"), Path("lena.jpeg"), Path("lena.bmp"),
        Path("Lena.png"), Path("Lena.jpg"), Path("Lena.jpeg"), Path("Lena.bmp"),
        Path("lenna.png"), Path("lenna.jpg"), Path("lenna.jpeg"), Path("lenna.bmp"),
    ]

    for path in candidate_paths:
        if path.exists():
            image = img_as_float(io.imread(path))
            break
    else:
        image = img_as_float(data.astronaut())

    if image.ndim == 3 and image.shape[-1] == 4:
        image = rgba2rgb(image)
    if image.ndim == 3:
        image = rgb2gray(image)
    return image


def convolve2d(image, kernel):
    """实现二维卷积。

    学生需要补全：
    1. 将 kernel 转为 float 类型的 numpy 数组。
    2. 按卷积定义翻转卷积核，即上下翻转、左右翻转。
    3. 根据卷积核大小计算上下左右需要填充的宽度。
    4. 使用 np.pad 对图像做 reflect 边界填充。
    5. 遍历原图每个像素位置，取出对应窗口。
    6. 计算窗口与卷积核对应元素乘积之和。
    7. 返回卷积结果。
    """
    # TODO：请补全二维卷积实现。
    raise NotImplementedError("请补全 convolve2d。")


def normalize_image(image):
    """将图像归一化到 [0, 1]，方便显示。"""
    image = np.asarray(image, dtype=float)
    min_value = image.min()
    max_value = image.max()
    if max_value == min_value:
        return np.zeros_like(image)
    return (image - min_value) / (max_value - min_value)


def show_images(images, titles, cols=3, cmap="gray"):
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 4.0 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


image = load_gray_image()
show_images([image], ["Original Image"], cols=1)

## 代码段 1：补全 Sobel、Prewitt、Roberts 和拉普拉斯算子

补全顺序建议：

1. 先补全 `gradient_by_kernels`，它负责根据 x/y 方向模板计算梯度幅值和方向。
2. 再补全 `sobel_gradient`、`prewitt_gradient` 和 `roberts_gradient` 中的卷积模板。
3. 最后补全 `laplacian_response`，根据 `kernel_type` 选择 4 邻域或 8 邻域拉普拉斯模板。

In [ ]:
def gradient_by_kernels(image, kernel_x, kernel_y):
    """根据两个方向模板计算梯度。

    学生需要补全：
    1. 使用 convolve2d 分别计算 gx 和 gy。
    2. 使用 sqrt(gx^2 + gy^2) 计算梯度幅值。
    3. 使用 np.arctan2(gy, gx) 计算梯度方向。
    4. 返回 gx、gy、magnitude、direction。
    """
    # TODO：请补全梯度计算。
    raise NotImplementedError("请补全 gradient_by_kernels。")


def sobel_gradient(image):
    """使用 Sobel 算子计算图像梯度。

    学生需要补全：
    1. 按理论部分写出 Sobel 的 x 方向模板。
    2. 按理论部分写出 Sobel 的 y 方向模板。
    3. 调用 gradient_by_kernels 返回结果。
    """
    # TODO：请补全 Sobel 模板和调用。
    raise NotImplementedError("请补全 sobel_gradient。")


def prewitt_gradient(image):
    """使用 Prewitt 算子计算图像梯度。

    学生需要补全：
    1. 写出 Prewitt 的 x 方向模板。
    2. 写出 Prewitt 的 y 方向模板。
    3. 调用 gradient_by_kernels 返回结果。
    """
    # TODO：请补全 Prewitt 模板和调用。
    raise NotImplementedError("请补全 prewitt_gradient。")


def roberts_gradient(image):
    """使用 Roberts 算子计算图像梯度。

    学生需要补全：
    1. 写出 Roberts 的两个 2 x 2 交叉差分模板。
    2. 调用 gradient_by_kernels 返回结果。
    """
    # TODO：请补全 Roberts 模板和调用。
    raise NotImplementedError("请补全 roberts_gradient。")


def laplacian_response(image, kernel_type=4):
    """使用拉普拉斯算子计算二阶边缘响应。

    学生需要补全：
    1. 当 kernel_type == 4 时，使用 4 邻域拉普拉斯模板。
    2. 当 kernel_type == 8 时，使用 8 邻域拉普拉斯模板。
    3. 对其他 kernel_type 抛出 ValueError。
    4. 调用 convolve2d 得到响应结果。
    """
    # TODO：请补全拉普拉斯响应计算。
    raise NotImplementedError("请补全 laplacian_response。")


sobel_gx, sobel_gy, sobel_mag, sobel_dir = sobel_gradient(image)
prewitt_gx, prewitt_gy, prewitt_mag, prewitt_dir = prewitt_gradient(image)
roberts_gx, roberts_gy, roberts_mag, roberts_dir = roberts_gradient(image)
laplace = laplacian_response(image, kernel_type=4)

show_images(
    [image, normalize_image(sobel_mag), normalize_image(prewitt_mag), normalize_image(roberts_mag), normalize_image(np.abs(laplace))],
    ["Original Image", "Sobel Magnitude", "Prewitt Magnitude", "Roberts Magnitude", "Laplacian Response"],
    cols=3
)

## 代码段 2：查看各算子的方向梯度

补全前一个代码段后，运行下面代码观察不同算子的 x 方向梯度、y 方向梯度和梯度幅值。

In [ ]:
show_images(
    [
        normalize_image(np.abs(sobel_gx)), normalize_image(np.abs(sobel_gy)), normalize_image(sobel_mag),
        normalize_image(np.abs(prewitt_gx)), normalize_image(np.abs(prewitt_gy)), normalize_image(prewitt_mag),
        normalize_image(np.abs(roberts_gx)), normalize_image(np.abs(roberts_gy)), normalize_image(roberts_mag),
    ],
    [
        "Sobel Gx", "Sobel Gy", "Sobel Magnitude",
        "Prewitt Gx", "Prewitt Gy", "Prewitt Magnitude",
        "Roberts Gx", "Roberts Gy", "Roberts Magnitude",
    ],
    cols=3
)

## 代码段 3：补全经典 Canny 边缘检测

请按 Canny 的完整流程补全下面函数。建议先单独测试每一个步骤，再运行最终的 `canny_edge_detection`。

In [ ]:
def gaussian_kernel(kernel_size=5, sigma=1.2):
    """生成二维高斯核。

    学生需要补全：
    1. 检查 kernel_size 是否为正奇数。
    2. 生成以 0 为中心的 x、y 坐标网格。
    3. 根据二维高斯函数计算每个位置的权重。
    4. 将高斯核归一化，使所有元素之和为 1。
    """
    # TODO：请补全高斯核生成。
    raise NotImplementedError("请补全 gaussian_kernel。")


def non_maximum_suppression(magnitude, direction):
    """非极大值抑制，使边缘变细。

    学生需要补全：
    1. 将弧度方向转换为角度，并映射到 [0, 180)。
    2. 遍历非边界像素。
    3. 根据方向将角度近似分成 0、45、90、135 度四类。
    4. 沿对应方向比较当前像素与前后两个邻居的梯度幅值。
    5. 如果当前像素是局部最大值则保留，否则置为 0。
    """
    # TODO：请补全非极大值抑制。
    raise NotImplementedError("请补全 non_maximum_suppression。")


def double_threshold(image, low_ratio=0.08, high_ratio=0.18):
    """双阈值检测。

    学生需要补全：
    1. 根据图像最大响应值和 high_ratio 计算高阈值。
    2. 根据高阈值和 low_ratio 计算低阈值。
    3. 大于等于高阈值的像素标记为强边缘 strong = 1.0。
    4. 位于低阈值和高阈值之间的像素标记为弱边缘 weak = 0.5。
    5. 小于低阈值的像素标记为 0。
    6. 返回阈值结果、weak 和 strong。
    """
    # TODO：请补全双阈值检测。
    raise NotImplementedError("请补全 double_threshold。")


def hysteresis(edge_map, weak=0.5, strong=1.0):
    """滞后边缘连接。

    学生需要补全：
    1. 遍历弱边缘像素。
    2. 检查它的 8 邻域内是否存在强边缘。
    3. 如果与强边缘连通，则将该弱边缘提升为强边缘。
    4. 重复检查，直到没有新的弱边缘被提升。
    5. 删除所有未连接到强边缘的弱边缘。
    """
    # TODO：请补全滞后边缘连接。
    raise NotImplementedError("请补全 hysteresis。")


def canny_edge_detection(image, gaussian_size=5, sigma=1.2, low_ratio=0.08, high_ratio=0.18):
    """完整 Canny 边缘检测流程。

    学生需要补全：
    1. 调用 gaussian_kernel 生成高斯核，并使用 convolve2d 平滑图像。
    2. 调用 sobel_gradient 计算梯度幅值和方向。
    3. 调用 non_maximum_suppression 得到细化后的边缘响应。
    4. 调用 double_threshold 得到强弱边缘图。
    5. 调用 hysteresis 得到最终边缘。
    6. 返回用于显示的中间结果和最终结果。
    """
    # TODO：请补全完整 Canny 流程。
    raise NotImplementedError("请补全 canny_edge_detection。")


# ===== 可调实验参数 =====
gaussian_size = 5
sigma = 1.2
low_ratio = 0.08
high_ratio = 0.18

smoothed, canny_mag, suppressed, thresholded, canny_edges = canny_edge_detection(
    image,
    gaussian_size=gaussian_size,
    sigma=sigma,
    low_ratio=low_ratio,
    high_ratio=high_ratio
)

show_images(
    [image, smoothed, canny_mag, suppressed, thresholded, canny_edges],
    ["Original Image", "Gaussian Smoothed", "Gradient Magnitude", "Non-Max Suppression", "Double Threshold", "Canny Edges"],
    cols=3
)

## 检查建议

完成代码后，可以按下面方式检查：

- `convolve2d` 能对简单矩阵和简单卷积核给出合理结果。
- Sobel 和 Prewitt 的梯度幅值图应突出主要轮廓边缘。
- Roberts 的边缘通常更细，但也可能更敏感。
- 拉普拉斯响应中，灰度突变位置应较明显。
- Canny 最终边缘图应比单纯梯度幅值图更细、更清晰。
- 调整 `sigma`、`low_ratio` 和 `high_ratio`，观察边缘数量和连续性的变化。